# YOLOv8 Food Detection Fine-tuning (Multi-Dataset Merge)

여러 Roboflow 프로젝트의 데이터셋을 병합하여 YOLOv8n을 fine-tune하고 ONNX로 export합니다.

**기존 클래스가 사라지는 문제(Catastrophic Forgetting)를 방지하기 위해 모든 데이터셋을 하나로 합쳐서 학습합니다.**

**런타임 → 런타임 유형 변경 → GPU(T4)** 로 설정 후 실행하세요.

## 1. 환경 설정

In [ ]:
!pip install ultralytics roboflow pyyaml -q

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. 데이터셋 다운로드

Roboflow에서 여러 프로젝트의 데이터셋을 다운로드합니다.

### 데이터셋 목록 설정

`DATASETS` 리스트에 사용할 데이터셋을 추가하세요.
각 항목은 `(workspace, project, version)` 튜플입니다.

In [ ]:
from roboflow import Roboflow

# === Roboflow API 키를 입력하세요 ===
API_KEY = "YOUR_API_KEY"

# === 병합할 데이터셋 목록 ===
# (workspace, project, version) 튜플로 추가
DATASETS = [
    # 기존 데이터셋 (40종)
    ("anonymous-fhuzh", "korean-food-iaryc", 1),
    # 새 데이터셋 (30종) — 여기에 추가
    # ("workspace-name", "project-name", version_number),
]

rf = Roboflow(api_key=API_KEY)

downloaded = []
for workspace, project_name, version_num in DATASETS:
    print(f'\n--- Downloading: {workspace}/{project_name} v{version_num} ---')
    project = rf.workspace(workspace).project(project_name)
    version = project.version(version_num)
    dataset = version.download("yolov8")
    downloaded.append(dataset)
    print(f'  Location: {dataset.location}')

print(f'\n총 {len(downloaded)}개 데이터셋 다운로드 완료')

## 3. 데이터셋 병합

각 데이터셋의 클래스를 통합 클래스 맵으로 재매핑하고, 이미지와 라벨을 하나의 디렉터리에 병합합니다.

In [ ]:
import os
import yaml
import shutil
from pathlib import Path

MERGED_DIR = Path('/content/merged_dataset')


def load_class_names(dataset_location):
    """data.yaml에서 클래스 이름 목록을 읽어옵니다."""
    yaml_path = os.path.join(dataset_location, 'data.yaml')
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    names = data.get('names', [])
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names.keys())]
    return names


# 1) 모든 데이터셋에서 클래스 이름을 수집하여 통합 클래스 맵 생성
unified_names = []  # 통합 클래스 리스트 (순서 = ID)
dataset_class_maps = []  # 각 데이터셋별 {원래ID -> 통합ID} 매핑

for ds in downloaded:
    names = load_class_names(ds.location)
    class_map = {}
    for old_id, name in enumerate(names):
        # 중복 클래스는 기존 ID 재사용
        if name not in unified_names:
            unified_names.append(name)
        new_id = unified_names.index(name)
        class_map[old_id] = new_id
    dataset_class_maps.append(class_map)
    print(f'{ds.location}: {len(names)} classes -> mapped to unified IDs')

print(f'\n통합 클래스 수: {len(unified_names)}')
for i, name in enumerate(unified_names):
    print(f'  {i}: {name}')

In [ ]:
def remap_labels(src_label_path, dst_label_path, class_map):
    """YOLO 라벨 파일의 class ID를 통합 맵에 맞게 재매핑합니다."""
    with open(src_label_path, 'r') as f:
        lines = f.readlines()

    remapped = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        old_class = int(parts[0])
        if old_class not in class_map:
            continue
        parts[0] = str(class_map[old_class])
        remapped.append(' '.join(parts))

    with open(dst_label_path, 'w') as f:
        f.write('\n'.join(remapped) + '\n')


def merge_split(split, dataset_idx, dataset, class_map):
    """하나의 split(train/valid/test)을 병합 디렉터리에 복사합니다."""
    src_img_dir = Path(dataset.location) / split / 'images'
    src_lbl_dir = Path(dataset.location) / split / 'labels'

    if not src_img_dir.exists():
        return 0

    dst_img_dir = MERGED_DIR / split / 'images'
    dst_lbl_dir = MERGED_DIR / split / 'labels'
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    count = 0
    for img_file in src_img_dir.iterdir():
        if img_file.suffix.lower() not in ('.jpg', '.jpeg', '.png', '.bmp', '.webp'):
            continue

        # 파일명 충돌 방지: ds{index}_{원래파일명}
        prefix = f'ds{dataset_idx}_'
        new_img_name = prefix + img_file.name
        new_lbl_name = prefix + img_file.stem + '.txt'

        # 이미지 복사
        shutil.copy2(img_file, dst_img_dir / new_img_name)

        # 라벨 재매핑 후 복사
        src_lbl = src_lbl_dir / (img_file.stem + '.txt')
        if src_lbl.exists():
            remap_labels(src_lbl, dst_lbl_dir / new_lbl_name, class_map)
        else:
            # 라벨 없는 이미지 (negative sample) — 빈 라벨 생성
            (dst_lbl_dir / new_lbl_name).touch()

        count += 1
    return count


# 기존 병합 디렉터리 초기화
if MERGED_DIR.exists():
    shutil.rmtree(MERGED_DIR)

# 각 데이터셋을 순회하면서 병합
for idx, (ds, class_map) in enumerate(zip(downloaded, dataset_class_maps)):
    print(f'\n--- Merging dataset {idx}: {ds.location} ---')
    for split in ['train', 'valid', 'test']:
        n = merge_split(split, idx, ds, class_map)
        if n > 0:
            print(f'  {split}: {n} images')

# 통합 data.yaml 생성
merged_yaml = {
    'train': str(MERGED_DIR / 'train' / 'images'),
    'val': str(MERGED_DIR / 'valid' / 'images'),
    'nc': len(unified_names),
    'names': unified_names,
}

# test split이 있으면 추가
test_dir = MERGED_DIR / 'test' / 'images'
if test_dir.exists() and any(test_dir.iterdir()):
    merged_yaml['test'] = str(test_dir)

yaml_path = MERGED_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(merged_yaml, f, default_flow_style=False, allow_unicode=True)

print(f'\n병합 완료!')
print(f'  총 클래스: {len(unified_names)}')
for split in ['train', 'valid', 'test']:
    img_dir = MERGED_DIR / split / 'images'
    if img_dir.exists():
        print(f'  {split}: {len(list(img_dir.iterdir()))} images')
print(f'  data.yaml: {yaml_path}')

### 병합 결과 검증

In [ ]:
import random
from collections import Counter

# 라벨 분포 확인
class_counts = Counter()
label_dir = MERGED_DIR / 'train' / 'labels'

for lbl_file in label_dir.iterdir():
    with open(lbl_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_counts[int(parts[0])] += 1

print(f'Train 라벨 분포 ({sum(class_counts.values())} annotations):')
print('-' * 40)
for class_id in sorted(class_counts.keys()):
    name = unified_names[class_id] if class_id < len(unified_names) else '???'
    print(f'  [{class_id:3d}] {name:30s} {class_counts[class_id]:5d}')

# 클래스 누락 확인
missing = [i for i in range(len(unified_names)) if i not in class_counts]
if missing:
    print(f'\n주의: train에 없는 클래스: {[unified_names[i] for i in missing]}')
else:
    print(f'\n모든 {len(unified_names)}개 클래스에 학습 데이터 있음')

## 4. YOLOv8n Fine-tune

병합된 데이터셋으로 학습합니다.

**BASE_MODEL 선택:**
- `yolov8n.pt`: COCO pretrained 베이스 (처음부터 학습)
- 기존 `best.pt`: 이전에 학습한 모델 베이스 (추가 학습, 더 빠름)

In [ ]:
from ultralytics import YOLO

# === 베이스 모델 선택 ===
# 처음부터 학습하려면 'yolov8n.pt'
# 기존 모델에서 이어서 학습하려면 기존 best.pt 경로
BASE_MODEL = 'yolov8n.pt'

model = YOLO(BASE_MODEL)

results = model.train(
    data=str(MERGED_DIR / 'data.yaml'),
    epochs=80,
    imgsz=640,
    batch=16,
    patience=15,
    device=0,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    project='food_detect',
    name='yolov8n_merged',
)

## 5. 학습 결과 확인

In [ ]:
from IPython.display import Image, display

result_dir = 'food_detect/yolov8n_merged'

# 학습 곡선
display(Image(filename=f'{result_dir}/results.png', width=800))

In [ ]:
# Confusion matrix
display(Image(filename=f'{result_dir}/confusion_matrix.png', width=600))

In [ ]:
# Validation 샘플
display(Image(filename=f'{result_dir}/val_batch0_pred.jpg', width=800))

## 6. ONNX Export

In [ ]:
best_model = YOLO(f'{result_dir}/weights/best.pt')

best_model.export(
    format='onnx',
    opset=13,
    simplify=True,
)

print(f'\nExport 완료: {result_dir}/weights/best.onnx')

In [ ]:
import onnx, os

onnx_path = f'{result_dir}/weights/best.onnx'
m = onnx.load(onnx_path)

print(f'Opset: {m.opset_import[0].version}')
print(f'Input: {m.graph.input[0].name} {[d.dim_value or d.dim_param for d in m.graph.input[0].type.tensor_type.shape.dim]}')
print(f'Output: {m.graph.output[0].name} {[d.dim_value or d.dim_param for d in m.graph.output[0].type.tensor_type.shape.dim]}')
print(f'Size: {os.path.getsize(onnx_path) / 1024 / 1024:.1f} MB')

## 7. 라벨 목록 출력

앱의 `labels.ts`를 교체할 라벨 배열을 생성합니다.

In [ ]:
names = best_model.names

print(f'// {len(names)} classes')
print('export const LABELS: string[] = [')
for idx in sorted(names.keys()):
    print(f"\t'{names[idx]}',")
print('];')

## 8. 다운로드

In [ ]:
from google.colab import files

files.download(f'{result_dir}/weights/best.onnx')
files.download(f'{result_dir}/weights/best.pt')

## 다음 단계

1. 다운로드한 `best.onnx`를 `static/model.onnx`로 교체
2. 위에서 출력된 라벨 배열로 `src/lib/onnx/labels.ts` 교체
3. `src/lib/onnx/session.ts`의 `NUM_CLASSES`를 새 클래스 수로 변경
4. `npm run dev`로 테스트

### 추가 데이터셋 병합 시
1. `DATASETS` 리스트에 새 데이터셋 추가
2. 이 노트북을 처음부터 다시 실행
3. 클래스 재매핑이 자동으로 처리됨